In [0]:
raw_path = "abfss://tokyo-olympic-data@tokyoolympicstorageacc.dfs.core.windows.net/raw-data"

athletes = spark.read.csv(f"{raw_path}/athletes.csv", header=True, inferSchema=True)
coaches = spark.read.csv(f"{raw_path}/coaches.csv", header=True, inferSchema=True)
entriesgender = spark.read.csv(f"{raw_path}/entriesgender.csv", header=True, inferSchema=True)
medals = spark.read.csv(f"{raw_path}/medals.csv", header=True, inferSchema=True)
teams = spark.read.csv(f"{raw_path}/teams.csv", header=True, inferSchema=True)

In [0]:
athletes.show(10)

In [0]:
athletes.printSchema()

In [0]:
coaches.printSchema()

In [0]:
entriesgender.printSchema()

In [0]:
medals.printSchema()

In [0]:
teams.printSchema()

In [0]:
from pyspark.sql import functions as F

def clean(df):
    # Trim whitespace in all string columns and drop exact duplicate rows
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(f"`{c}`")))
    return df.dropDuplicates()

# Athletes: trim + dedupe (1 duplicate row in raw data)
athletes_t = clean(athletes)

# Coaches: trim + dedupe; Event is missing for team sports -> label it explicitly
coaches_t = clean(coaches).fillna({"Event": "Not Specified"})

# Entries by gender: add female / male share of total entries
entriesgender_t = (clean(entriesgender)
    .withColumn("Female_Pct", F.round(F.col("Female") / F.col("Total") * 100, 2))
    .withColumn("Male_Pct", F.round(F.col("Male") / F.col("Total") * 100, 2)))

# Medals: rename columns with '/' and spaces so they are valid for Parquet/Delta
medals_t = (clean(medals)
    .withColumnRenamed("Team/NOC", "Team_NOC")
    .withColumnRenamed("Rank by Total", "Rank_by_Total")
    .orderBy("Rank"))

# Teams: trim + dedupe
teams_t = clean(teams)

for name, df in [("athletes", athletes_t), ("coaches", coaches_t), ("entriesgender", entriesgender_t), ("medals", medals_t), ("teams", teams_t)]:
    print(name, df.count(), df.columns)

In [0]:
from pyspark.sql.types import IntegerType, StringType, DecimalType

# Enforce explicit types instead of relying on inferSchema

# Text columns -> string
for name in ["athletes_t", "coaches_t", "teams_t"]:
    df = globals()[name]
    globals()[name] = df.select([F.col(c).cast(StringType()).alias(c) for c in df.columns])

# Entries by gender: counts -> int, percentages -> decimal(5,2)
entriesgender_t = (entriesgender_t
    .withColumn("Discipline", F.col("Discipline").cast(StringType()))
    .withColumn("Female", F.col("Female").cast(IntegerType()))
    .withColumn("Male", F.col("Male").cast(IntegerType()))
    .withColumn("Total", F.col("Total").cast(IntegerType()))
    .withColumn("Female_Pct", F.col("Female_Pct").cast(DecimalType(5, 2)))
    .withColumn("Male_Pct", F.col("Male_Pct").cast(DecimalType(5, 2))))

# Medals: ranks and medal counts -> int, team -> string
medals_t = medals_t.withColumn("Team_NOC", F.col("Team_NOC").cast(StringType()))
for c in ["Rank", "Gold", "Silver", "Bronze", "Total", "Rank_by_Total"]:
    medals_t = medals_t.withColumn(c, F.col(c).cast(IntegerType()))

# Check: casting must not create nulls
for name, df in [("athletes", athletes_t), ("coaches", coaches_t), ("entriesgender", entriesgender_t), ("medals", medals_t), ("teams", teams_t)]:
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    print(name, df.dtypes, "nulls:", {k: v for k, v in nulls.items() if v})

In [0]:
transformed_path = "abfss://tokyo-olympic-data@tokyoolympicstorageacc.dfs.core.windows.net/transformed-data"

outputs = {
    "athletes": athletes_t,
    "coaches": coaches_t,
    "entriesgender": entriesgender_t,
    "medals": medals_t,
    "teams": teams_t,
}

# One folder per dataset, single CSV file with header (data is small)
for name, df in outputs.items():
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{transformed_path}/{name}")

# Verify: read back and compare row counts
for name, df in outputs.items():
    written = spark.read.csv(f"{transformed_path}/{name}", header=True).count()
    print(f"{name}: expected {df.count()}, written {written}")